# 02 · Soil water storage groups (Iowa)

**Direction:** Soil and Crop Resilience Planning. **User:** Soil Conservation Office.
**Decision:** where soil programs would cut drought losses most.

This notebook prepares and checks the **storage groups** (low / middle / high `aws_100cm_mm`) for Iowa corn and Iowa soybeans. It does **not** estimate any drought-loss effect; that comes later.

1. **Step 1** — understand `aws_100cm_mm` in Iowa and define the groups.
2. **Step 2** — check the groups are fair to compare: soil-quality confounding, drought exposure, heat, and geography.

Run with the kernel's working directory in `notebooks/` (what `jupyter nbconvert --execute notebooks/02_storage_groups.ipynb` from the repo root does).

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data_loader import load_master, load_soil, load_counties

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")

17,056 rows × 21 columns


## Column names and thresholds

All column names below were verified against `data/data_dictionary.md`. Soil fields in the master are `aws_100cm_mm`, `droughty_pct`, `nccpi_crop`; the soil panel (`load_soil()`) also carries `nccpi_corn`, `nccpi_soy`, `nccpi_sg`.

In [2]:
# ---- Column names (master dataset) ----
COL_FIPS = "fips"
COL_COUNTY = "county_name"
COL_YEAR = "year"
COL_STATE = "state_alpha"
COL_CROP = "crop"
COL_YIELD = "yield_per_acre"
COL_YIELD_STATUS = "yield_status"
COL_ANOM = "yield_anomaly_pct"
COL_SEV = "max_drought_severity"       # 0-5; 3 = D2, 4 = D3, 5 = D4
COL_WEEKS_D2 = "weeks_in_d2_plus"
COL_HEAT = "extreme_heat_days"
COL_PRECIP_ANOM = "precip_anomaly_pct"
COL_AWS = "aws_100cm_mm"               # soil water storage, static per county
COL_DROUGHTY = "droughty_pct"          # % of county on drought-vulnerable soil, static per county
COL_NCCPI = "nccpi_crop"               # soil productivity index, matched to the row's crop

# ---- Drought coding ----
SEV_D2 = 3
SEV_D3 = 4

# ---- Scope ----
FOCUS_STATE = "IA"
FOCUS_CROPS = ["CORN", "SOYBEANS"]

# ---- Storage grouping ----
# N_GROUPS = 3 -> bottom / middle / top third of Iowa counties by aws_100cm_mm (tertiles)
# N_GROUPS = 2 -> median split (low / high)
N_GROUPS = 3
GROUP_LABELS = {3: ["low", "middle", "high"], 2: ["low", "high"]}[N_GROUPS]
GROUP_COLORS = {"low": "#d7301f", "middle": "#fdd49e", "high": "#2b8cbe"}
COL_GROUP = "storage_group"

In [3]:
ia = df[df[COL_STATE] == FOCUS_STATE].copy()
assert ia[COL_FIPS].map(type).eq(str).all() and ia[COL_FIPS].str.len().eq(5).all(), "fips must be 5-char text"
print(f"Iowa rows: {len(ia):,}  counties: {ia[COL_FIPS].nunique()}  crops: {sorted(ia[COL_CROP].unique())}")
print(ia.groupby(COL_CROP).size())

Iowa rows: 5,876  counties: 99  crops: ['CORN', 'SOYBEANS', 'WHEAT']
crop
CORN        2574
SOYBEANS    2574
WHEAT        728
dtype: int64


---
## Step 1 — Understand `aws_100cm_mm` in Iowa

### Is it static per county?

In [4]:
distinct_aws = ia.groupby(COL_FIPS)[COL_AWS].nunique(dropna=False)
print("Distinct aws_100cm_mm values per county (count of counties):")
print(distinct_aws.value_counts().sort_index().rename_axis("distinct_values").rename("n_counties"))
assert (distinct_aws == 1).all(), "aws_100cm_mm varies within a county"

print("\nMissing aws_100cm_mm in Iowa rows:", int(ia[COL_AWS].isna().sum()),
      "| counties with NaN:", int(ia.groupby(COL_FIPS)[COL_AWS].apply(lambda s: s.isna().any()).sum()))
print("Missing droughty_pct in Iowa rows:", int(ia[COL_DROUGHTY].isna().sum()))
print("Missing nccpi_crop in Iowa rows:  ", int(ia[COL_NCCPI].isna().sum()))

# Does nccpi_crop differ by crop within a county?
nccpi_per_county = ia.groupby(COL_FIPS)[COL_NCCPI].nunique()
nccpi_per_county_crop = ia.groupby([COL_FIPS, COL_CROP])[COL_NCCPI].nunique()
print("\nnccpi_crop distinct values per county ACROSS crops:", nccpi_per_county.value_counts().sort_index().to_dict())
print("nccpi_crop distinct values per county WITHIN a crop:", nccpi_per_county_crop.value_counts().sort_index().to_dict())
print("-> nccpi_crop is static within county x crop but differs between crops, so it is kept per crop below.")

Distinct aws_100cm_mm values per county (count of counties):
distinct_values
1    99
Name: n_counties, dtype: int64

Missing aws_100cm_mm in Iowa rows: 0 | counties with NaN: 0
Missing droughty_pct in Iowa rows: 0
Missing nccpi_crop in Iowa rows:   0

nccpi_crop distinct values per county ACROSS crops: {1: 1, 2: 70, 3: 28}
nccpi_crop distinct values per county WITHIN a crop: {1: 226}
-> nccpi_crop is static within county x crop but differs between crops, so it is kept per crop below.


### County-level table (one row per Iowa county)

`nccpi_crop` is kept per crop as `nccpi_CORN` and `nccpi_SOYBEANS` (pivoted from the master). The soil panel's `nccpi_corn` / `nccpi_soy` are merged in as a cross-check.

In [5]:
county = (ia.groupby(COL_FIPS)
            .agg(county_name=(COL_COUNTY, "first"),
                 aws_100cm_mm=(COL_AWS, "first"),
                 droughty_pct=(COL_DROUGHTY, "first"))
            .reset_index())

nccpi_wide = (ia[ia[COL_CROP].isin(FOCUS_CROPS)]
                .groupby([COL_FIPS, COL_CROP])[COL_NCCPI].first()
                .unstack(COL_CROP))
nccpi_wide.columns = [f"nccpi_{c}" for c in nccpi_wide.columns]
county = county.merge(nccpi_wide.reset_index(), on=COL_FIPS, how="left")

# Cross-check against the soil panel
soil = load_soil()
soil_ia = soil[soil[COL_FIPS].str.startswith("19")]
county = county.merge(soil_ia[[COL_FIPS, "nccpi_corn", "nccpi_soy", "nccpi_sg"]], on=COL_FIPS, how="left")
# The master rounds nccpi_crop to 4 decimals (build_master.py); the soil panel keeps full precision.
assert np.allclose(county["aws_100cm_mm"], soil_ia.set_index(COL_FIPS).loc[county[COL_FIPS], "aws_100cm_mm"].values)
assert np.allclose(county["nccpi_CORN"], county["nccpi_corn"], atol=1e-4)
assert np.allclose(county["nccpi_SOYBEANS"], county["nccpi_soy"], atol=1e-4)
print("Master soil fields match the soil panel for all 99 Iowa counties (nccpi to 4 decimals).")
county = county.drop(columns=["nccpi_corn", "nccpi_soy"])   # redundant with the per-crop columns
print(county.shape)
county.head(10)

Master soil fields match the soil panel for all 99 Iowa counties (nccpi to 4 decimals).
(99, 7)


,fips,county_name,aws_100cm_mm,droughty_pct,nccpi_CORN,nccpi_SOYBEANS,nccpi_sg
0,19001,Adair County,181.112470,11.519473,0.7842,0.6629,0.603554
1,19003,Adams County,177.608667,19.323799,0.6764,0.5709,0.548239
2,19005,Allamakee County,170.589367,25.015396,0.5752,0.4782,0.500035
3,19007,Appanoose County,162.187838,25.906222,0.6676,0.5604,0.565623
4,19009,Audubon County,189.507205,4.000703,0.8148,0.6752,0.597011
5,19011,Benton County,172.398106,9.683993,0.8317,0.7070,0.630376
6,19013,Black Hawk County,161.461328,34.689853,0.7722,0.6608,0.620463
7,19015,Boone County,179.215432,4.098584,0.6929,0.7266,0.641136
8,19017,Bremer County,154.245463,48.351121,0.7522,0.6503,0.619029
9,19019,Buchanan County,162.170523,42.782443,0.7938,0.6864,0.644499


In [6]:
print("aws_100cm_mm across 99 Iowa counties:")
display(county["aws_100cm_mm"].describe().round(2).to_frame().T)
print("\nOther county-level soil fields:")
display(county[["droughty_pct", "nccpi_CORN", "nccpi_SOYBEANS", "nccpi_sg"]].describe().round(3).T)

fig = px.histogram(county, x="aws_100cm_mm", nbins=25,
                   title="Iowa counties: available water storage, top 100 cm (mm)",
                   labels={"aws_100cm_mm": "aws_100cm_mm (mm)"})
fig.update_layout(bargap=0.05, yaxis_title="counties")
fig.show()

aws_100cm_mm across 99 Iowa counties:


,count,mean,std,min,25%,50%,75%,max
aws_100cm_mm,99.0,179.24,11.42,154.25,172.0,179.0,184.78,205.76



Other county-level soil fields:


,count,mean,std,min,25%,50%,75%,max
droughty_pct,99.0,14.946,11.584,0.132,4.060,14.050,22.327,48.351
nccpi_CORN,99.0,0.741,0.055,0.575,0.714,0.742,0.775,0.886
nccpi_SOYBEANS,99.0,0.674,0.067,0.478,0.632,0.675,0.724,0.794
nccpi_sg,99.0,0.621,0.043,0.482,0.593,0.624,0.661,0.693


### Define storage groups

Bottom / middle / top third of the 99 counties by `aws_100cm_mm` (`N_GROUPS = 3`). Set `N_GROUPS = 2` at the top for a median split.

In [7]:
county[COL_GROUP] = pd.qcut(county["aws_100cm_mm"], q=N_GROUPS, labels=GROUP_LABELS)
county[COL_GROUP] = county[COL_GROUP].astype(str)

cut_edges = county["aws_100cm_mm"].quantile(np.linspace(0, 1, N_GROUPS + 1)).round(2)
print("Cutoffs (quantile edges of aws_100cm_mm):")
print(cut_edges.to_string())
print()
grp = (county.groupby(COL_GROUP)
             .agg(n_counties=(COL_FIPS, "size"),
                  aws_min=("aws_100cm_mm", "min"),
                  aws_mean=("aws_100cm_mm", "mean"),
                  aws_max=("aws_100cm_mm", "max"))
             .reindex(GROUP_LABELS).round(2))
display(grp)

# Attach the group to every Iowa row for the exposure checks below
ia = ia.merge(county[[COL_FIPS, COL_GROUP]], on=COL_FIPS, how="left")
assert ia[COL_GROUP].notna().all()

Cutoffs (quantile edges of aws_100cm_mm):
0.000000    154.25
0.333333    174.07
0.666667    182.70
1.000000    205.76



,n_counties,aws_min,aws_mean,aws_max
storage_group,,,,
low,33,154.25,167.26,173.81
middle,33,174.20,178.73,182.70
high,33,182.72,191.74,205.76


---
## Step 2 — Are the groups fair to compare?

### Storage vs soil quality

In [8]:
cols = ["aws_100cm_mm", "droughty_pct", "nccpi_CORN", "nccpi_SOYBEANS", "nccpi_sg"]
print("Pearson correlation across the 99 Iowa counties:")
display(county[cols].corr(method="pearson").round(3))
print("Spearman (rank) correlation:")
display(county[cols].corr(method="spearman").round(3))

for c in ["nccpi_CORN", "nccpi_SOYBEANS"]:
    r = county["aws_100cm_mm"].corr(county[c])
    rho = county["aws_100cm_mm"].corr(county[c], method="spearman")
    print(f"aws_100cm_mm vs {c}: Pearson r = {r:.3f}, Spearman rho = {rho:.3f}")

fig = px.scatter(county, x="aws_100cm_mm", y="nccpi_CORN", color=COL_GROUP,
                 color_discrete_map=GROUP_COLORS, category_orders={COL_GROUP: GROUP_LABELS},
                 hover_name="county_name", trendline="ols",
                 title="Iowa counties: soil water storage vs corn soil productivity (NCCPI)")
fig.show()

Pearson correlation across the 99 Iowa counties:


,aws_100cm_mm,droughty_pct,nccpi_CORN,nccpi_SOYBEANS,nccpi_sg
aws_100cm_mm,1.000,-0.831,0.069,0.467,0.178
droughty_pct,-0.831,1.000,-0.150,-0.575,-0.375
nccpi_CORN,0.069,-0.150,1.000,0.504,0.609
nccpi_SOYBEANS,0.467,-0.575,0.504,1.000,0.866
nccpi_sg,0.178,-0.375,0.609,0.866,1.000


Spearman (rank) correlation:


,aws_100cm_mm,droughty_pct,nccpi_CORN,nccpi_SOYBEANS,nccpi_sg
aws_100cm_mm,1.000,-0.878,0.089,0.533,0.251
droughty_pct,-0.878,1.000,-0.090,-0.633,-0.425
nccpi_CORN,0.089,-0.090,1.000,0.344,0.433
nccpi_SOYBEANS,0.533,-0.633,0.344,1.000,0.868
nccpi_sg,0.251,-0.425,0.433,0.868,1.000


aws_100cm_mm vs nccpi_CORN: Pearson r = 0.069, Spearman rho = 0.089
aws_100cm_mm vs nccpi_SOYBEANS: Pearson r = 0.467, Spearman rho = 0.533


### Every soil-related column available

In [9]:
master_soil_cols = [COL_AWS, COL_DROUGHTY, COL_NCCPI]
print("Soil columns in the master:", master_soil_cols)
print("Columns in load_soil():    ", list(soil.columns))
print()
print("What they are (from the data dictionary):")
print("  aws_100cm_mm  - available water supply in top 100 cm, area-weighted (mm). Static.")
print("  droughty_pct  - % of county area on soil with <= 152 mm storage. A threshold view of the SAME idea as aws.")
print("  nccpi_crop    - NCCPI productivity index matched to the row's crop (corn / soy / small grains; sorghum uses corn).")
print("  nccpi_sg      - NCCPI small grains (wheat). Not relevant to corn/soy rows.")
print()
print("The soil panel has no other properties (no texture, slope, organic matter, drainage class). droughty_pct is")
print("not an independent confounder: it is derived from the same AWS map. nccpi_* is the only separate soil property,")
print("and it is the one the data dictionary names as the control variable.")

Soil columns in the master: ['aws_100cm_mm', 'droughty_pct', 'nccpi_crop']
Columns in load_soil():     ['fips', 'state_alpha', 'aws_100cm_mm', 'droughty_pct', 'nccpi_corn', 'nccpi_soy', 'nccpi_sg']

What they are (from the data dictionary):
  aws_100cm_mm  - available water supply in top 100 cm, area-weighted (mm). Static.
  droughty_pct  - % of county area on soil with <= 152 mm storage. A threshold view of the SAME idea as aws.
  nccpi_crop    - NCCPI productivity index matched to the row's crop (corn / soy / small grains; sorghum uses corn).
  nccpi_sg      - NCCPI small grains (wheat). Not relevant to corn/soy rows.

The soil panel has no other properties (no texture, slope, organic matter, drainage class). droughty_pct is
not an independent confounder: it is derived from the same AWS map. nccpi_* is the only separate soil property,
and it is the one the data dictionary names as the control variable.


### Low vs high: soil quality, heat and drought exposure (reported-yield rows, by crop)

In [10]:
rep = ia[(ia[COL_CROP].isin(FOCUS_CROPS)) & ia[COL_YIELD].notna()].copy()
rep["is_d2plus"] = rep[COL_SEV] >= SEV_D2
rep["is_d3plus"] = rep[COL_SEV] >= SEV_D3

cmp = (rep.groupby([COL_CROP, COL_GROUP])
          .agg(n_counties=(COL_FIPS, "nunique"),
               county_years=(COL_FIPS, "size"),
               mean_nccpi_crop=(COL_NCCPI, "mean"),
               mean_aws=(COL_AWS, "mean"),
               mean_extreme_heat_days=(COL_HEAT, "mean"),
               share_D2plus=("is_d2plus", "mean"),
               share_D3plus=("is_d3plus", "mean"),
               mean_weeks_d2plus=(COL_WEEKS_D2, "mean"),
               mean_precip_anom_pct=(COL_PRECIP_ANOM, "mean"))
          .reindex(pd.MultiIndex.from_product([FOCUS_CROPS, GROUP_LABELS], names=[COL_CROP, COL_GROUP]))
          .round(3))
display(cmp)

print("Low minus high, by crop:")
diff = (cmp.xs("low", level=COL_GROUP) - cmp.xs("high", level=COL_GROUP))[
    ["mean_nccpi_crop", "mean_extreme_heat_days", "share_D2plus", "share_D3plus", "mean_weeks_d2plus"]].round(3)
display(diff)

n_counties  county_years  mean_nccpi_crop  mean_aws  mean_extreme_heat_days  share_D2plus  share_D3plus  mean_weeks_d2plus  mean_precip_anom_pct
crop     storage_group                                                                                                                                                  
CORN     low                    33           820            0.732   167.334                   2.179         0.290         0.122              2.094                 0.408
         middle                 33           824            0.752   178.737                   2.108         0.316         0.106              2.098                 0.025
         high                   33           826            0.742   191.633                   2.591         0.320         0.109              2.426                -0.367
SOYBEANS low                    33           817            0.637   167.335                   2.225         0.313         0.141              3.094                 0.050
         middle                 33           809            0.674   178.782                   2.131         0.347         0.119              3.106                 0.118
         high                   33           811            0.717   191.539                   2.572         0.369         0.131              3.291                -0.149

Low minus high, by crop:


,mean_nccpi_crop,mean_extreme_heat_days,share_D2plus,share_D3plus,mean_weeks_d2plus
crop,,,,,
CORN,-0.01,-0.412,-0.030,0.013,-0.332
SOYBEANS,-0.08,-0.347,-0.056,0.010,-0.197


In [11]:
# Same years? Share of reported counties at D2+ in each year, low vs high group (corn and soybeans).
yr = (rep.groupby([COL_CROP, COL_YEAR, COL_GROUP])["is_d2plus"].mean()
         .unstack(COL_GROUP)[GROUP_LABELS].round(2))
yr.columns = [f"D2+ share: {g}" for g in yr.columns]
heat_yr = (rep.groupby([COL_CROP, COL_YEAR, COL_GROUP])[COL_HEAT].mean()
              .unstack(COL_GROUP)[["low", "high"]].round(1))
heat_yr.columns = [f"heat days: {g}" for g in heat_yr.columns]
by_year = yr.join(heat_yr)
display(by_year.loc["CORN"])
display(by_year.loc["SOYBEANS"])

# Year-level agreement: in the years where anyone was D2+, how different was low vs high exposure?
for crop in FOCUS_CROPS:
    t = by_year.loc[crop]
    active = t[(t["D2+ share: low"] > 0) | (t["D2+ share: high"] > 0)]
    gap = (active["D2+ share: low"] - active["D2+ share: high"])
    print(f"{crop}: {len(active)} years with any D2+. Low-minus-high D2+ share: mean {gap.mean():+.2f}, "
          f"range {gap.min():+.2f} to {gap.max():+.2f}. Years where |gap| >= 0.25: "
          f"{list(gap[gap.abs() >= 0.25].index)}")

,D2+ share: low,D2+ share: middle,D2+ share: high,heat days: low,heat days: high
year,,,,,
2000,0.76,0.76,1.00,2.0,3.5
2001,0.00,0.00,0.00,0.6,1.2
2002,0.03,0.12,0.27,1.1,3.7
2003,0.97,0.97,0.85,4.1,5.1
2004,0.00,0.00,0.00,0.0,0.0
2005,0.42,0.36,0.00,2.9,1.7
2006,0.39,0.45,0.70,3.5,5.3
2007,0.00,0.09,0.21,0.5,1.1
2008,0.00,0.00,0.00,0.1,0.4


,D2+ share: low,D2+ share: middle,D2+ share: high,heat days: low,heat days: high
year,,,,,
2000,0.76,0.76,1.00,2.0,3.5
2001,0.00,0.00,0.00,0.6,1.2
2002,0.03,0.12,0.27,1.1,3.7
2003,0.97,0.97,0.85,4.1,5.1
2004,0.00,0.00,0.00,0.0,0.0
2005,0.61,0.61,0.21,2.9,1.7
2006,0.39,0.45,0.70,3.5,5.3
2007,0.00,0.09,0.21,0.5,1.1
2008,0.00,0.00,0.00,0.1,0.4


CORN: 17 years with any D2+. Low-minus-high D2+ share: mean -0.04, range -0.82 to +0.43. Years where |gap| >= 0.25: [2005, 2006, 2018, 2020, 2024]
SOYBEANS: 17 years with any D2+. Low-minus-high D2+ share: mean -0.10, range -0.79 to +0.42. Years where |gap| >= 0.25: [2005, 2006, 2018, 2020, 2022]


### Maps: storage and storage group

Counties with no Iowa crop rows would be blank; all 99 Iowa counties have corn and soybean rows, so none should be.

In [12]:
counties_geo = load_counties()
ia_geo = {"type": "FeatureCollection",
          "features": [f for f in counties_geo["features"] if str(f["id"]).startswith("19")]}
print("Iowa features in counties.geojson:", len(ia_geo["features"]))

fig_aws = px.choropleth(county, geojson=ia_geo, locations=COL_FIPS, color="aws_100cm_mm",
                        color_continuous_scale="YlGnBu", hover_name="county_name",
                        hover_data={COL_FIPS: True, "aws_100cm_mm": ":.1f", COL_GROUP: True})
fig_grp = px.choropleth(county, geojson=ia_geo, locations=COL_FIPS, color=COL_GROUP,
                        color_discrete_map=GROUP_COLORS, category_orders={COL_GROUP: GROUP_LABELS},
                        hover_name="county_name", hover_data={COL_FIPS: True, "aws_100cm_mm": ":.1f"})

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "geo"}, {"type": "geo"}]],
                    subplot_titles=("aws_100cm_mm (mm)", f"storage group ({N_GROUPS} groups)"))
for tr in fig_aws.data:
    fig.add_trace(tr, row=1, col=1)
for tr in fig_grp.data:
    fig.add_trace(tr, row=1, col=2)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(coloraxis=fig_aws.layout.coloraxis, height=420, width=1000,
                  margin=dict(l=10, r=10, t=50, b=10),
                  title_text="Iowa: soil water storage (left) and storage group (right)",
                  legend=dict(title="storage group", x=0.78, y=0.05))
fig.update_coloraxes(colorbar=dict(x=0.46, len=0.8, title="mm"))
fig.show()

Iowa features in counties.geojson: 99


### Do low-storage counties cluster geographically?

Rough county centroids are computed from the GeoJSON polygon vertices, then each county is placed into a 3 × 3 grid of the state (west / central / east by longitude, north / central / south by latitude, using terciles of the centroids).

In [13]:
def centroid(feature):
    geom = feature["geometry"]
    polys = geom["coordinates"] if geom["type"] == "Polygon" else [p for mp in geom["coordinates"] for p in mp]
    pts = np.array([pt for ring in polys for pt in ring])
    return pts[:, 0].mean(), pts[:, 1].mean()

cent = pd.DataFrame([(str(f["id"]), *centroid(f)) for f in ia_geo["features"]],
                    columns=[COL_FIPS, "lon", "lat"])
county = county.merge(cent, on=COL_FIPS, how="left")
assert county[["lon", "lat"]].notna().all().all()

county["ew"] = pd.qcut(county["lon"], 3, labels=["west", "central", "east"]).astype(str)
county["ns"] = pd.qcut(county["lat"], 3, labels=["south", "central", "north"]).astype(str)

print("Mean centroid by storage group:")
display(county.groupby(COL_GROUP)[["lon", "lat", "aws_100cm_mm"]].mean().reindex(GROUP_LABELS).round(2))

print("\nCounties per (north-south band x east-west band) for the LOW group:")
display(pd.crosstab(county.loc[county[COL_GROUP] == "low", "ns"], county.loc[county[COL_GROUP] == "low", "ew"])
          .reindex(index=["north", "central", "south"], columns=["west", "central", "east"], fill_value=0))
print("Counties per band for the HIGH group:")
display(pd.crosstab(county.loc[county[COL_GROUP] == "high", "ns"], county.loc[county[COL_GROUP] == "high", "ew"])
          .reindex(index=["north", "central", "south"], columns=["west", "central", "east"], fill_value=0))

print("\nCorrelation of aws_100cm_mm with longitude:", round(county["aws_100cm_mm"].corr(county["lon"]), 3),
      "| with latitude:", round(county["aws_100cm_mm"].corr(county["lat"]), 3))

Mean centroid by storage group:


,lon,lat,aws_100cm_mm
storage_group,,,
low,-92.54,41.95,167.26
middle,-92.93,41.97,178.73
high,-95.02,42.23,191.74



Counties per (north-south band x east-west band) for the LOW group:


ew,west,central,east
ns,,,
north,0,5,8
central,0,2,3
south,0,8,7


Counties per band for the HIGH group:


ew,west,central,east
ns,,,
north,9,2,0
central,10,6,0
south,6,0,0



Correlation of aws_100cm_mm with longitude: -0.668 | with latitude: 0.111


In [14]:
# The low- and high-storage counties by name, for reference
for g in ["low", "high"]:
    sub = county[county[COL_GROUP] == g].sort_values("aws_100cm_mm")
    print(f"{g.upper()} storage ({len(sub)} counties): " +
          ", ".join(f"{n.replace(' County', '')} ({a:.0f})" for n, a in zip(sub["county_name"], sub["aws_100cm_mm"])))
    print()

LOW storage (33 counties): Bremer (154), Wayne (157), Floyd (159), Black Hawk (161), Buchanan (162), Appanoose (162), Decatur (163), Butler (163), Davis (163), Clarke (164), Lucas (165), Mitchell (165), Linn (165), Chickasaw (166), Van Buren (166), Monroe (167), Ringgold (167), Howard (169), Louisa (170), Lee (171), Allamakee (171), Delaware (171), Muscatine (172), Hamilton (172), Marion (172), Fayette (172), Benton (172), Jefferson (173), Worth (173), Wapello (173), Webster (174), Wright (174), Dubuque (174)

HIGH storage (33 counties): Story (183), Hancock (183), Greene (183), Hardin (183), Dallas (184), Grundy (184), Monona (185), Clay (185), Marshall (185), Polk (185), Emmet (186), Buena Vista (187), Osceola (187), Winnebago (189), Cass (189), Audubon (190), Lyon (190), Carroll (190), Page (190), Sac (190), Montgomery (191), Fremont (197), Cherokee (198), Sioux (199), Harrison (199), Woodbury (199), Mills (200), Pottawattamie (201), O'Brien (201), Shelby (202), Ida (202), Crawford 

---
## Notes for the write-up

- `aws_100cm_mm` is one value per county (static), with no missing values for any of the 99 Iowa counties.
- `nccpi_crop` is static within county × crop but differs between corn and soybeans, so it is kept per crop.
- The only soil properties available are AWS, droughty_pct (derived from AWS) and NCCPI. NCCPI is the only independent soil confounder to control for.
- No drought-loss effect is estimated here.